In [ ]:
"""
1. 梯度累积、清零与计算图复用
使用：
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3
完成以下操作：
- 对同一个 y 连续反向传播两次，正确使用 retain_graph=True。
- 打印每次的 x.grad。
- 清零梯度，再对同一个 y 反向传播一次。
- 随后重新计算 new_loss = 5 * x，不清零就反向传播，观察梯度。
- 再重新计算一次 5 * x，这次先清零，只得到新损失的梯度。
解释：保留计算图、重新前向计算、清零梯度，分别解决什么问题？
"""

import torch

x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

y.backward(retain_graph=True)
print(x.grad)

y.backward(retain_graph=True)
print(x.grad)

x.grad.zero_()
print(x.grad)

y.backward()
print(x.grad)

new_loss = 5 * x
new_loss.backward(retain_graph=True)
print(x.grad)

x.grad.zero_()
new_loss = 5 * x
new_loss.backward()
print(x.grad)

"""
保留计算图: 让同一次前向计算得到的y可以再次反向传播. 通常第一次反向传播后, 反向计算所需的中间结果会被释放; 它不会清空x.grad.
重新前向计算: 重新执行5*x, 得到可用于新一次反向传播的计算图, 因此这一步不需要服用旧图.
清零梯度: 把已累积的x.grad清掉, 使下一次只包含新损失的梯度; 它不改变x的值, 也不负责重建计算图.
"""

In [ ]:
"""
2. 求和、平均与加权反向传播
每种情况都重新创建：
v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2
分别计算：
- out.sum().backward()。
- out.mean().backward()。
- out.backward(torch.tensor([1.0, 0.0, 2.0]))。
- 将上一种写成等价的加权标量表达式，再调用无参数 .backward()。
手算并验证四组梯度，解释：第三种为什么不是直接把 [1, 0, 2] 存进 v.grad？
"""

v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2
out.sum().backward()
print("sum: ", v.grad)
assert torch.allclose(v.grad, torch.tensor([2.0, 4.0, 6.0]))

v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2
out.mean().backward()
print("mean: ", v.grad)
assert torch.allclose(v.grad, torch.tensor([2.0/3, 4.0/3, 2.0]))

v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2
out.backward([torch.tensor([1.0, 0.0, 2.0])])
print("weighted backward: ", v.grad)
assert torch.allclose(v.grad, torch.tensor([2.0, 0.0, 12.0]))

v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2
(out[0] + 2 * out[2]).backward()
print("weighted scalar: ", v.grad)
assert torch.allclose(v.grad, torch.tensor([2.0, 0.0, 12.0]))
"""
第三种为什么不是直接把 [1, 0, 2] 存进 v.grad？
[1, 0, 2]是传给out各分量的反向传播权重, 不是v的最终梯度. 还要经过out=v²这一层的求导; 三个分量的导数是[2, 4, 6], 逐项乘权重后才得到[2, 0, 12].
"""

In [ ]:
"""
3. detach() 与 no_grad()
分别新建 x = torch.tensor(3.0, requires_grad=True)，比较：
# 情况 A
y = x ** 2
z = y * x

# 情况 B
y = x ** 2
u = y.detach()
z = u * x

# 情况 C
with torch.no_grad():
    u = x ** 2
z = u * x
每种情况都执行 z.backward()，打印梯度。
解释：为什么三种 z 的数值相同，梯度却不全相同？情况 C 中，x.requires_grad 是否被改成了 False？
"""

# 情况 A
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
z = y * x
z.backward()
print(x.grad)

# 情况 B
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
u = y.detach()
z = u * x
z.backward()
print(x.grad)

# 情况 C
x = torch.tensor(3.0, requires_grad=True)
with torch.no_grad():
    u = x ** 2
z = u * x
z.backward()
print(x.grad)

"""
解释：为什么三种 z 的数值相同，梯度却不全相同？
detach()切断了从u回到y=x²的路径; 反向传播仍通过z=u*x中那个直接使用的x, 计算的是dz/dx = 9.

情况 C 中，x.requires_grad 是否被改成了 False？
否, u.requires_grad被改成False, 因为这步发生在no_grad()内; no_grad()只临时停止记录其中的运算, 并未修改原有的x.requires_grad=True.
"""

In [ ]:
"""
4. 动态计算图与条件分支
实现：
def g(a):
    b = a
    for _ in range(2):
        b = b * 2
    if b.sum() > 0:
        return 100 * b
    else:
        return b ** 2
分别使用浮点输入 2.0、-2.0、[2.0, -2.0]，每次独立创建需要梯度的张量。
打印输出和输入梯度；多元素输出使用 .sum().backward()。
解释：向量输入时，是整个向量选择一个分支，还是每个元素分别选择分支？
"""
def g(a):
    b = a
    for _ in range(2):
        b = b * 2
    if b.sum() > 0:
        return 100 * b
    else:
        return b ** 2

for value in (2.0, -2.0, [2.0, -2.0]):
    a = torch.tensor(value, requires_grad=True)
    result = g(a)
    loss = result.sum() if result.numel() > 1 else result
    loss.backward()
    print("a: ", a, "\nresult: ", result, "\na.grad: ", a.grad)

"""
向量输入时，是整个向量选择一个分支，还是每个元素分别选择分支？
向量输入时是整个向量选一个分支. [8, -8]的和是0, 不满足严格的 > 0, 所以两个元素都进入平方分支; 不是正元素走上面, 负元素走下面. 向量结果先.sum(), 才能按题目要求用无参数的.backward().
"""

In [ ]:
"""
5. 二阶导数与正弦导数绘图
这一题分两小段：
- 对 x=3.0、y=x**3，用 torch.autograd.grad 求一阶、二阶导数，正确使用 create_graph=True，并手算验证。
- 在 [-π, π] 上取 201 个点，用 Autograd 求 sin(x) 的导数，将函数和导数画在同一张图中。
绘图可直接沿用我们之前的代码，不必重写，但要实际运行并保留图像。不能用 torch.cos(x) 代替自动求导。
"""

# 1. 二阶导数
x = torch.tensor(3.0, requires_grad=True)
y = x ** 3

d1, = torch.autograd.grad(y, x, create_graph=True)
d2, = torch.autograd.grad(d1, x)

print("一阶导数: ", d1)
print("二阶导数: ", d2)

assert torch.allclose(d1, torch.tensor(27.0))
assert torch.allclose(d2, torch.tensor(18.0))

# 2. 用Autograd绘制正弦及其导数
import math
import matplotlib.pyplot as plt
import torch

x = torch.linspace(-math.pi, math.pi, 201, requires_grad=True)
y = torch.sin(x)

dy, = torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y))

fig, ax = plt.subplots()
ax.plot(x.detach().cpu().numpy(), y.detach().cpu().numpy(), label="sin(x)")
ax.plot(x.detach().cpu().numpy(), dy.detach().cpu().numpy(), label="derivative from Autograd")
ax.axhline(0, color="gray", linewidth=0.8)
ax.legend()
ax.grid(True)

fig.savefig("sin_autograd_derivative.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
"""
6. 官方示例：从参数到损失，再到梯度
使用官方示例：
x = torch.ones(5)
y = torch.zeros(3)
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)

z = x @ w + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
loss.backward()
打印 z、loss、w.grad、b.grad 的形状，以及 z.grad_fn、loss.grad_fn。
解释三个问题：
- 为什么 x 不需要梯度，仍然可以计算 w.grad？
- 为什么这里 w.grad 的每一行都与 b.grad 相同？提示：x 的五个元素全是 1。
- .backward() 后，参数本身有没有被更新？
"""

x = torch.ones(5)
y = torch.zeros(3)
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)

w_before = w.detach().clone()
b_before = b.detach().clone()

z = x @ w + b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
loss.backward()

print("z.shape: ", z.shape)
print("loss.shape: ", loss.shape)
print("w.grad.shape: ", w.grad.shape)
print("b.grad.shape: ", b.grad.shape)
print("z.grad_fn: ", z.grad_fn)
print("loss.grad_fn: ", loss.grad_fn)

print("w.grad: ", w.grad)
print("b.grad: ", b.grad)
print("每行相同:", torch.allclose(w.grad, b.grad.expand_as(w.grad)))
print("w 未更新:", torch.equal(w, w_before))
print("b 未更新:", torch.equal(b, b_before))

"""
- 为什么 x 不需要梯度，仍然可以计算 w.grad？
z = x @ w + b用到了需要梯度的w和b. Autograd会记录这些运算, 从loss沿计算图回到w, b; 不要求所有参与运算的张量都设置requires_grad=True.
- 为什么这里 w.grad 的每一行都与 b.grad 相同？提示：x 的五个元素全是 1。
对任意一列j, z[j]中的w[i, j]乘着x[i], 而b[j]是直接相加. 因此该位置的权重梯度等于x[i] * b.grad[j]. 这里每个x[i]都是1, 所以w.grad的五行都等于b.grad.
- .backward() 后，参数本身有没有被更新？
没有. loss.backward()只计算并存储梯度; 之后调用optimizer.step() (或手动执行更新)才会改变参数

"""